# PhysVLM V2 Colab Deployment Notebook

Use this notebook for the V2 deployment loop. If the model checkpoint, PhysVLM source tree, and EQA-phys data have not been prepared yet, run `notebooks/01_colab_pro_reproduction.ipynb` first. This notebook starts after V1 reproduction assets already exist.

Target runtime: Colab Pro / Pro+ with A100 GPU.


## What You Need In Colab

You need two categories of files.

**A. This V2 reproduction repo**

Recommended after committing/pushing V2:

```bash
git clone https://github.com/Xxc33128/PhysVLM-reproduce.git
```

If the V2 changes are not pushed yet, upload one zip to Colab instead. From the parent directory that contains `PhysVLM-reproduce`, create:

```bash
zip -r PhysVLM-reproduce-v2.zip PhysVLM-reproduce \
  -x "*/.git/*" "*/__pycache__/*" "*/results/images/*" "*/results/raw/*" "*/artifacts/*" "*.zip" "*.tar" "*.tar.gz"
```

Upload `PhysVLM-reproduce-v2.zip` to `/content/`, or place it anywhere under Google Drive and use the auto-find cell below.

**B. Heavy runtime assets**

The official PhysVLM source and EQA simulator can be cloned/generated in this notebook under `/content/PhysVLM`. Keep model weights on Google Drive. The notebook searches these common checkpoint locations:

```text
/content/drive/MyDrive/physvlm/checkpoints/PhysVLM-Qwen2.5-3B
/content/drive/MyDrive/physvlm/PhysVLM-Qwen2.5-3B
/content/drive/MyDrive/PhysVLM/PhysVLM-Qwen2.5-3B
```

Do not upload model weights, ONNX models, TensorRT engines, or generated simulator images into this GitHub repo.


## 0. Mount Drive And Check GPU

Before running, set the Colab runtime to GPU and verify that the device is A100 or another CUDA GPU.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## 1. Get The V2 Repo Into Colab

Use `REPO_SOURCE = "github"` only after the V2 scripts have been pushed. Use `REPO_SOURCE = "zip"` when you uploaded `PhysVLM-reproduce-v2.zip` manually.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import zipfile

REPO_SOURCE = 'zip'  # change to 'github' after pushing V2
REPO_URL = 'https://github.com/Xxc33128/PhysVLM-reproduce.git'
ZIP_PATH = Path('/content/PhysVLM-reproduce-v2.zip')
REPO_DIR = Path('/content/PhysVLM-reproduce')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if REPO_SOURCE == 'github':
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
elif REPO_SOURCE == 'zip':
    if not ZIP_PATH.exists():
        candidates = []
        for root in [Path('/content'), Path('/content/drive/MyDrive')]:
            if root.exists():
                candidates.extend(root.rglob('*PhysVLM*reproduce*v2*.zip'))
                candidates.extend(root.rglob('*PhysVLM-reproduce*.zip'))
        candidates = sorted(set(candidates))
        print('Found zip candidates:')
        for item in candidates[:20]:
            print(' -', item)
        if not candidates:
            raise FileNotFoundError(f'Upload {ZIP_PATH} first, or set REPO_SOURCE="github" after pushing.')
        shutil.copy2(candidates[0], ZIP_PATH)
        print('Copied zip to', ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall('/content')
    if not REPO_DIR.exists():
        candidates = [p for p in Path('/content').iterdir() if p.is_dir() and p.name.startswith('PhysVLM-reproduce')]
        if not candidates:
            raise FileNotFoundError('Could not find extracted PhysVLM-reproduce folder under /content.')
        candidates[0].rename(REPO_DIR)
else:
    raise ValueError('REPO_SOURCE must be github or zip')

os.chdir(REPO_DIR)
print('repo:', Path.cwd())
!git status -sb || true


## 2. Install Runtime Dependencies

Colab usually already has CUDA PyTorch. Install the repo dependencies plus V2 deployment extras.


In [ ]:
!pip -q install -r requirements.txt
!pip -q install onnx onnxscript onnxruntime-gpu fastapi uvicorn pydantic requests


## 3. Prepare Official PhysVLM, Smoke Data, And Paths

This cell handles the pieces that are not inside the V2 repo zip:

- clone official PhysVLM to `/content/PhysVLM`
- apply the two V1 compatibility patches
- generate a small UR5 `val` dataset and S-P Maps if needed
- create `phys_bench_sim_qas_smoke.json`
- find the model checkpoint on Drive
- set `PHYSVLM_ROOT`, `PHYSVLM_MODEL_PATH`, `PHYSVLM_QA_JSON`, and `PHYSVLM_DATA_ROOT`

For a quick deployment smoke test, keep `GENERATE_FULL_4_ROBOTS = False`. Generate the full four-robot benchmark only after the deployment chain is stable.


In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
from tqdm import tqdm

GENERATE_FULL_4_ROBOTS = False
ROBOTS = ['CR5', 'FR5', 'UR5', 'PANDA'] if GENERATE_FULL_4_ROBOTS else ['UR5']

PHYSVLM_REPO = Path('/content/PhysVLM')
PHYSVLM_ROOT = PHYSVLM_REPO / 'physvlm-main'
SIM_ROOT = PHYSVLM_REPO / 'EQA-phys-simulator'
REPO_DIR = Path('/content/PhysVLM-reproduce')

MODEL_CANDIDATES = [
    Path('/content/drive/MyDrive/physvlm/checkpoints/PhysVLM-Qwen2.5-3B'),
    Path('/content/drive/MyDrive/physvlm/PhysVLM-Qwen2.5-3B'),
    Path('/content/drive/MyDrive/PhysVLM/PhysVLM-Qwen2.5-3B'),
]

def run(cmd, cwd=None):
    print('$', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, check=True)

if not PHYSVLM_ROOT.exists():
    if not PHYSVLM_REPO.exists():
        run(['git', 'clone', 'https://github.com/unira-zwj/PhysVLM.git', PHYSVLM_REPO])
    else:
        raise FileNotFoundError(f'{PHYSVLM_REPO} exists but {PHYSVLM_ROOT} does not exist.')

builder = PHYSVLM_ROOT / 'physvlm/model/builder.py'
text = builder.read_text()
patched_text, n = re.subn(
    r'(\n\s*print\("---model---\\n", model\)\n)\s*return None(\n\s*image_processor = None)',
    r'\1\2',
    text,
    count=1,
)
if n:
    builder.write_text(patched_text)
    print('Patched builder.py return-None bug')
else:
    print('builder.py return-None patch already applied or pattern not found')

encoder_builder = PHYSVLM_ROOT / 'physvlm/model/multimodal_encoder/builder.py'
text = encoder_builder.read_text()
if '_resolve_siglip_alias' not in text:
    text = text.replace(
        'from .siglip_encoder import SigLipVisionTower\n\n\n',
        'from .siglip_encoder import SigLipVisionTower\n\n\n'
        'def _resolve_siglip_alias(tower_name):\n'
        '    if tower_name == "Siglip/siglip-so400m-patch14-384":\n'
        '        return "google/siglip-so400m-patch14-384"\n'
        '    return tower_name\n\n\n',
    )
    text = text.replace(
        "vision_tower = getattr(vision_tower_cfg, 'mm_vision_tower', getattr(vision_tower_cfg, 'vision_tower', None))",
        "vision_tower = _resolve_siglip_alias(getattr(vision_tower_cfg, 'mm_vision_tower', getattr(vision_tower_cfg, 'vision_tower', None)))",
    )
    text = text.replace(
        "depth_tower = getattr(depth_tower_cfg, 'mm_depth_tower', getattr(depth_tower_cfg, 'depth_tower', None))",
        "depth_tower = _resolve_siglip_alias(getattr(depth_tower_cfg, 'mm_depth_tower', getattr(depth_tower_cfg, 'depth_tower', None)))",
    )
    encoder_builder.write_text(text)
    print('Patched SigLIP alias')
else:
    print('SigLIP alias patch already applied')

for robot in ROBOTS:
    robot_dir = SIM_ROOT / 'val' / robot
    inrange_json = SIM_ROOT / 'val' / f'{robot}_inrange.json'
    rgb_files = list(robot_dir.glob('*rgb*.jpg')) if robot_dir.exists() else []
    if not inrange_json.exists() or not rgb_files:
        run(['python', 'main.py', '--robot', robot, '--dataset', 'val'], cwd=SIM_ROOT)
    else:
        print(f'[skip] {robot} RGB/metadata already exists')

    sp_map_files = list(robot_dir.glob('*sp_map*.jpg')) if robot_dir.exists() else []
    if not sp_map_files:
        run(['python', 'generate_sp_map.py', '--robot', robot], cwd=SIM_ROOT)
    else:
        print(f'[skip] {robot} S-P maps already exist')

if GENERATE_FULL_4_ROBOTS:
    run(['python', 'generate_qas.py'], cwd=SIM_ROOT)
    qa_json = SIM_ROOT / 'phys_bench_sim_qas.json'
else:
    qa_json = SIM_ROOT / 'phys_bench_sim_qas_smoke.json'
    final_labels = []
    for robot in ROBOTS:
        in_range_path = SIM_ROOT / 'val' / f'{robot}_inrange.json'
        with open(in_range_path, 'r') as f:
            inrange_labels = json.load(f)
        for item in tqdm(inrange_labels, desc=f'QA {robot}'):
            image_name = item['image']
            for cls in item['true']:
                for question in [
                    f"Is the {cls} in the robot's reachable space?",
                    f"Can the robot directly pick the {cls}?",
                ]:
                    final_labels.append({
                        'id': image_name.split('.')[0],
                        'image': os.path.join('val', robot, image_name),
                        'depth': os.path.join('val', robot, image_name.replace('rgb', 'sp_map')),
                        'question': question,
                        'answer': 'Yes',
                    })
            for cls in item['false']:
                for question in [
                    f"Is the {cls} in the robot's reachable space?",
                    f"Can the robot directly pick the {cls}?",
                ]:
                    final_labels.append({
                        'id': image_name.split('.')[0],
                        'image': os.path.join('val', robot, image_name),
                        'depth': os.path.join('val', robot, image_name.replace('rgb', 'sp_map')),
                        'question': question,
                        'answer': 'No',
                    })
    qa_json.write_text(json.dumps(final_labels, indent=2), encoding='utf-8')
    print('smoke QA count:', len(final_labels))

model_path = next((path for path in MODEL_CANDIDATES if path.exists()), None)
if model_path is None:
    print('Model checkpoint not found in common locations. Run this to find it:')
    print('!find /content/drive/MyDrive -maxdepth 6 -type d -iname "*PhysVLM*Qwen*" -print')
    raise FileNotFoundError('Set PHYSVLM_MODEL_PATH manually after locating the checkpoint.')

os.environ['PHYSVLM_ROOT'] = str(PHYSVLM_ROOT)
os.environ['PHYSVLM_MODEL_PATH'] = str(model_path)
os.environ['PHYSVLM_QA_JSON'] = str(qa_json)
os.environ['PHYSVLM_DATA_ROOT'] = str(SIM_ROOT)
os.environ['PYTHONPATH'] = f"{PHYSVLM_ROOT}:{os.environ.get('PYTHONPATH', '')}"

os.chdir(REPO_DIR)
print('\n=== V2 env ===')
for key in ['PHYSVLM_ROOT', 'PHYSVLM_MODEL_PATH', 'PHYSVLM_QA_JSON', 'PHYSVLM_DATA_ROOT']:
    value = os.environ[key]
    print(f'{key}: {value} -> exists={Path(value).exists()}')


## 4. Smoke Test: 5 Examples

Run these first. Do not start with 50 or 100 examples. The TensorRT EP step may fail because of environment or unsupported operators; that failure is still useful if it is captured in JSON.


In [ ]:
!python scripts/benchmark_torch_compile.py --limit 5 --output-json results/deployment/backend_results/torch_compile_smoke.json


In [ ]:
!python scripts/export_onnx_modules.py --module vision_tower --output-dir artifacts/onnx


In [ ]:
!python scripts/check_export_parity.py   --module vision_tower   --output-json results/deployment/backend_results/parity_vision_tower_smoke.json


In [ ]:
!python scripts/benchmark_onnx.py   --module vision_tower   --limit 5   --output-json results/deployment/backend_results/onnx_cuda_smoke.json


In [ ]:
!python scripts/benchmark_tensorrt_ep.py   --module vision_tower   --limit 5   --output-json results/deployment/backend_results/onnx_trt_ep_smoke.json


## 5. Formal 50-Example Run

Only run this after the smoke tests produce usable JSON files. These commands write the main backend result files expected by the deployment report.


In [ ]:
RUN_FULL_50 = False  # set True after smoke tests look correct

if RUN_FULL_50:
    !python scripts/benchmark_torch_compile.py --limit 50 --output-json results/deployment/backend_results/torch_compile.json
    !python scripts/benchmark_onnx.py --module vision_tower --limit 50 --output-json results/deployment/backend_results/onnx_cuda.json
    # TensorRT EP may fall back to CUDA if TensorRT libraries are missing. Keep formal TensorRT results separate.
else:
    print('Set RUN_FULL_50=True after smoke tests pass.')


## 6. Optional: Multimodal Projector Export

This broadens the deployment story from one exportable module to two. It uses synthetic hidden-state tensors because the projector input is an intermediate feature tensor.


In [ ]:
RUN_PROJECTOR = False

if RUN_PROJECTOR:
    !python scripts/export_onnx_modules.py --module mm_projector --output-dir artifacts/onnx
    !python scripts/check_export_parity.py --module mm_projector --output-json results/deployment/backend_results/parity_mm_projector.json
    !python scripts/benchmark_onnx.py --module mm_projector --synthetic --limit 50 --output-json results/deployment/backend_results/onnx_cuda_mm_projector.json
    !python scripts/benchmark_tensorrt_ep.py --module mm_projector --synthetic --limit 50 --output-json results/deployment/backend_results/onnx_trt_ep_mm_projector.json
else:
    print('Set RUN_PROJECTOR=True after vision_tower export is stable.')


## 7. FastAPI Serving Benchmark

This starts the API in the background, waits for `/health`, then runs sequential and concurrency-2 request benchmarks. API latency must be reported separately from pure model latency.


In [ ]:
RUN_FASTAPI = False

if RUN_FASTAPI:
    import subprocess, time, requests
    from pathlib import Path

    log_path = Path('/content/physvlm_api.log')
    server = subprocess.Popen(
        ['python', 'scripts/serve_physvlm.py', '--host', '0.0.0.0', '--port', '8000'],
        stdout=log_path.open('w'),
        stderr=subprocess.STDOUT,
    )
    print('server pid:', server.pid)

    health_url = 'http://127.0.0.1:8000/health'
    for attempt in range(90):
        try:
            response = requests.get(health_url, timeout=5)
            if response.ok:
                print(response.json())
                break
        except Exception:
            pass
        time.sleep(5)
    else:
        print(log_path.read_text()[-4000:])
        raise RuntimeError('FastAPI server did not become healthy.')

    !python scripts/benchmark_api.py --limit 10 --base-url http://127.0.0.1:8000 --output-json results/deployment/backend_results/fastapi.json
else:
    print('Set RUN_FASTAPI=True when you are ready to test serving.')


## 8. Inspect And Package Results

After the run, save the lightweight result JSON/report files. Do not package ONNX models or TensorRT engines into Git.


In [ ]:
!find results/deployment -maxdepth 3 -type f -print | sort
!find artifacts/onnx -maxdepth 1 -name '*.metadata.json' -print 2>/dev/null | sort || true


In [ ]:
from pathlib import Path

!tar -czf /content/physvlm_v2_deployment_results.tar.gz \
  results/deployment \
  artifacts/onnx/*.metadata.json 2>/dev/null || true

!ls -lh /content/physvlm_v2_deployment_results.tar.gz

drive_candidates = [Path('/content/drive/MyDrive/physvlm'), Path('/content/drive/MyDrive/PhysVLM')]
drive_out = next((path for path in drive_candidates if path.exists()), drive_candidates[0])
drive_out.mkdir(parents=True, exist_ok=True)
!cp /content/physvlm_v2_deployment_results.tar.gz {drive_out}/physvlm_v2_deployment_results.tar.gz
print('Copied to', drive_out / 'physvlm_v2_deployment_results.tar.gz')


## 9. What To Do Back In The Repo

Bring back `physvlm_v2_deployment_results.tar.gz` or the JSON files under `results/deployment/backend_results/`. Then update:

- `results/deployment/deployment_summary.csv`
- `results/deployment/deployment_report.md`
- README deployment conclusion if the numbers are stable

Keep large files out of Git: `.onnx`, TensorRT engines/plans, timing caches, model weights, and full simulator images.
